## Writing Gremlin queries

Since you can write Typescript code. This also means you can import modules and
use them. We use gremlin module to write the gremlin query using typescript sdk.

Please note that we don't use websockets here. Internally the traversal is
represented as bytecode in the Gremlin JS SDK, then translated locally into a
Gremlin string, and that string is sent through the SDR REST query API for
execution against Neptune. So the code that gets executed is still a gremlin
query in string format but it's still helpful since we can just write the full
Typescript Code here. It's helpful for quick prototyping. Since you can just
copy any code from your editor and paste it here to run a quick test and it'll
work without you needing to change any semantics.

In [1]:
import { connectNotebook, ui } from "@sdr-notebook/mod";
import { process as gremlinProcess } from "gremlin";

const client = await connectNotebook({ profile: "dsoadev" });
const g = client.g();
const _ = gremlinProcess.statics;

const studies = await client.gremlin<Array<Record<string, unknown>>>(
  g.V().hasLabel("Study").as("study").limit(3)
    .out("has_latest_version").as("latestVersion")
    .out("has_design").as("design")
    .project(
      "trial",
      "description",
      "createdAt",
      "latestVersion",
      "studyType",
      "collaborators",
      "totalActivities",
    )
    .by(_.select("study").values("id"))
    .by(_.select("study").values("description"))
    .by(_.select("study").values("createdAt"))
    .by(_.select("latestVersion").values("versionIdentifier"))
    .by(_.select("design").values("instanceType"))
    .by(
      _.select("latestVersion").out("has_source_version").out(
        "has_collaborator",
      ).values("email").fold(),
    )
    .by(_.select("design").out("has_activity").count()),
);

const mappedStudies = studies.map(({
  trial,
  description,
  latestVersion,
  studyType,
  collaborators,
  totalActivities,
  createdAt,
}) => ({
  trial,
  description,
  latestVersion,
  studyType,
  collaborators,
  totalActivities,
  createdAt,
}));
ui.table(mappedStudies);

trial,description,latestVersion,studyType,collaborators,totalActivities,createdAt
QU8-US-AKBZ,Prospective registry quantifying COPD exacerbations and biomarker changes.,1.0.0,ObservationalStudyDesign,"[ ""respiratory.analytics@fakesponsor.com"", ""pulmonary.site@fakesponsor.com"" ]",6,2025-11-27T16:14:23.447Z
NWS-UW-IGSB,Testing DSoA in Dev,1.0.0,InterventionalStudyDesign,"[ ""pai_shalini@lilly.com"", ""jyotsna.raghuraman@network.lilly.com"", ""greg.lara@network.lilly.com"", ""jana_samit@lilly.com"", ""jennings_donald@lilly.com"", ""josh.geddes@network.lilly.com"", ""pralhadbharmal.jadhav@network.lilly.com"" ]",17,2025-11-28T08:30:46.440Z
CRD-09-2036,Prospective registry quantifying COPD exacerbations and biomarker changes.,1.0.178,ObservationalStudyDesign,"[ ""dr.smith@hospital-networks.org"", ""sarah.jones@pharma-global.net"", ""data.manager@trials-inc.com"" ]",6,2025-11-28T05:27:51.262Z


There are some queries that do lend themselves nicely to Cypher than Gremlin.
Here's usecase.

Suppose we want to find out top 5 collaborators by number of studies. Here's
what we'll have to write if we do this via gremlin.

Notice the usage of a special operator called Scope.local.

Gremlin has two scopes that control what a step operates on:

Scope.global (the default) — the step operates on the entire traversal stream,
treating all traversers together as one collection. Scope.local — the step
operates inside each individual traverser's current object, treating it as a
local collection.

This can be tricky for new users who expect it work like regular SQL. For
instance if we remove Scope.local from limit and just do .limit(5) intead then
it would actually return more than 5 rows. So that limit doesn't actually apply
the the map that we have already collected after the order by clause. It applies
to the number of traversars. So in this instance there's only once traverse so
you still get the whole map ( instead of just 5 you'd normally expect it to
return)

## Traversals

When you write g.V(), Gremlin creates one traverser per vertex in the graph. If
you have 289 studies, g.V().hasLabel('Study') produces 289 traversers - one
pointing at each Study node. Think of them like little cursors, each
independently walking the graph as the query progresses.

Think of them like little cursors, each independently walking the graph as the
query progresses. So at any point in a query, the "traversal stream" is just the
current collection of all active traversers.

So when we do `g.V().hasLabel('Study').limit(5)` then it simply means cutting
the stream down to first 5 so we get 5 Study vertices back.

This is fine but this becomes important when use use group

`g.V().hasLabel('Collaborator') .group() .by('email') .by(...)

    `

.group() is a reducing step. It takes all 289 traversers and collapses them into
one single traverser holding one big Map. The stream went from 289 things to 1
thing.

Now if you call .limit(5) (global)

There's only 1 traverser in the stream (the map), so we get... that 1 map. All
of it. Limit did nothing useful.To cut inside that map, we need to tell Gremlin
to go local:

limit(Scope.local, 5) instead of just global limit like .limit(5)

So here's the basic idea:

*** A traverser is a cursor on a single object. limit (global) cuts how many
cursors exist. limit(Scope.local) cuts what's inside the object each cursor is
pointing at.***

Once .group(), .fold(), or .aggregate() collapses your many cursors into one
cursor holding a collection, all your slicing and sorting needs to switch to
Scope.local to have any effect.

In [7]:
const studies = await client.gremlin(`
  g.V().hasLabel('Collaborator')
  .group()
    .by('email')
    .by(__.in('has_collaborator').in('has_source_version').in('has_version').hasLabel('Study').dedup().count())
    .order(Scope.local).by(values, Order.desc)
  .limit(Scope.local, 5)`);

console.log(studies);

[
  {
    "pooja.e@network.lilly.com": 104,
    "dhanasekar.murugan@network.lilly.com": 85,
    "kayyala.sreelatha@network.lilly.com": 75,
    "jyotsna.raghuraman@network.lilly.com": 73,
    "jana_samit@lilly.com": 69
  }
]


Now let's compare the above query to its cypher equivalent.

This looks a lot simpler and intuitive to follow. Again, this is just subjective
but this atleast looks a bit more simpler to me and I don't have to deal with
different behaviour of a same operator in different contexts.

In [16]:
const studies = await client.cypher(`
    MATCH (s:Study)-[:has_version]->(:StudyVersion)-[:has_source_version]->(:USDMSource)-[:has_collaborator]->(c:Collaborator)
    WITH c.email AS collaborator, COUNT(DISTINCT s) AS studyCount
    ORDER BY studyCount DESC
    LIMIT 5
    RETURN collect([collaborator, studyCount]) AS stats
 `);

console.log(studies);

[
  {
    stats: [
      [ "pooja.e@network.lilly.com", 104 ],
      [ "dhanasekar.murugan@network.lilly.com", 85 ],
      [ "kayyala.sreelatha@network.lilly.com", 75 ],
      [ "jyotsna.raghuraman@network.lilly.com", 73 ],
      [ "jana_samit@lilly.com", 69 ]
    ]
  }
]


## Implementing audit log functionality

The idea is we want to log user activity every time they do something. So we
model it like following in our graph database:

1. We create a User node that has id, email and name. These fields should be
   there in the jwt token
2. We create a UserActivity node that contains action, time, ip and userAgent
   field

Here's how the flow works:

As soon as the request comes in - we just check if the user with this id already
exists. If they do then we create UserActivity Node and link it to existing User
node or we create a new User node and then a new UserActivity Node and link them
together.

In [33]:
import { connectNotebook, ui } from "@sdr-notebook/mod";
import { process as gremlinProcess } from "gremlin";
import { faker } from "npm:@faker-js/faker";

const client = await connectNotebook({ profile: "dsoadev" });
const g = client.g();
const _ = gremlinProcess.statics;

const actions = [
  "login",
  "logout",
  "accessTrialHistory",
  "accessTrialDetails",
  "deleteVersion",
  "exportTrial",
];

const userData = [];

for (let i = 0; i < 3; i++) {
  const user = {
    id: crypto.randomUUID(),
    name: faker.person.fullName(),
    email: faker.internet.email(),
  };

  for (let i = 0; i < 50; i++) {
    const activity = {
      action: actions[Math.floor(Math.random() * actions.length)],
      ip: faker.internet.ipv4(),
      userAgent: faker.internet.userAgent(),
    };
    userData.push({
      user,
      activity,
    });
  }
}

await Promise.all(userData.map(({ user, activity }) =>
  client.gremlin<unknown>(
    g
      .V()
      .hasLabel("User")
      .has("id", user.id)
      .fold()
      .coalesce(
        _.unfold(),
        _.addV("User")
          .property(gremlinProcess.cardinality.single, "id", user.id)
          .property(gremlinProcess.cardinality.single, "email", user.email)
          .property(
            gremlinProcess.cardinality.single,
            "createdAt",
            new Date().toISOString(),
          ),
      ).as("user")
      .addE("has_activity")
      .to(
        _.addV("UserActivity")
          .property(
            gremlinProcess.cardinality.single,
            "action",
            activity.action,
          )
          .property(gremlinProcess.cardinality.single, "ip", activity.ip)
          .property(
            gremlinProcess.cardinality.single,
            "userAgent",
            activity.userAgent,
          ),
      ),
  )
));

const logs = await client.gremlin<Array<Record<string, unknown>>>(
  g.V().hasLabel("User").as("users").project("user", "logs").by(
    _.select("users").elementMap().fold(),
  ).by(_.out("has_activity").elementMap().fold()),
);
console.log(logs);

CredentialsProviderError: TooManyRequestsException: HTTP 429 Unknown Code

In [29]:
const logs = await client.gremlin<Array<Record<string, unknown>>>(
  g.V().hasLabel("User").as("users").project("user", "logs").by(
    _.select("users").elementMap().fold(),
  ).by(_.out("has_activity").elementMap().fold()),
);
ui.json(logs);

CredentialsProviderError: TooManyRequestsException: HTTP 429 Unknown Code